# Solar Filament Segmentation Challenge 2026 — full pipeline from the released weights

This notebook regenerates the submitted predictions (`release/submission_final.csv`) **without retraining**: it loads the 10 released weights (5 folds × {seed‑1 SWA, seed 2}) and runs

`16‑view TTA per fold → vote score (α=0.75) → 5‑fold fusion → post‑processing (thr 0.32, area ≥ 200, retain fraction ≥ 0.8) → CSV`.

Set `N_IMAGES = None` in the settings cell for the full run (about 10 h on one 16 GB GPU, 41 s per test image). The default `N_IMAGES = 3` runs the same code on 3 test images (about 10 minutes) as a check. Requirements: `requirements.txt`, detectron2 0.6 and Mask DINO with `patches/maskdino_filament.patch` (see `docs/REPRODUCE.md`).

In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(REPO); sys.path.insert(0, str(REPO / "src"))
N_IMAGES = 3          # None = all test images (full reproduction, ~10 h)
ALPHA, SCORE_THR, MIN_AREA, KEEP_FRAC = 0.75, 0.32, 200, 0.8
WD, OUT = REPO / "release" / "weights", REPO / "release" / "run"
OUT.mkdir(parents=True, exist_ok=True)
def run(cmd):
    print(">", " ".join(str(c) for c in cmd)); subprocess.run([str(c) for c in cmd], check=True)

## 1. Weights
Each `.pth` is stored in parts below GitHub's 100 MB limit; `assemble` joins them and checks the SHA‑256 from `MANIFEST.json`.

In [ ]:
from weights_io import assemble
W = {(k, s): assemble(WD, f"fold{k}_{s}.pth") for k in range(5) for s in ("seed1_swa", "seed2")}
print(len(W), "weight files verified")

## 2. Test inference, per fold
Two models per fold × 8 D4 views = 16 views, merged by probability averaging with a topology guard; the number of agreeing views is stored with each instance (`--dump-raw`).

In [ ]:
raws = []
for k in range(5):
    raw = OUT / f"test_raw_f{k}.json"
    if not raw.exists():
        cmd = ["python3", "-u", "src/predict_submit.py", "--config", f"configs/_seed2_f{k}.yaml",
               "--weights", W[(k, "seed1_swa")], W[(k, "seed2")],
               "--tta", "--tta-views", 8, "--tta-merge", "prob", "--mask-bin-thr", 0.5,
               "--dump-raw", raw, "--out", OUT / f"tmp_f{k}.csv"]
        if N_IMAGES: cmd += ["--limit", N_IMAGES]
        subprocess.run([str(c) for c in cmd])      # exit code 1 only means the partial CSV failed the full-set check
    raws.append(raw)

## 3. Vote score and five‑fold fusion
`score' = score · (votes / 16)^α`; fold outputs are clustered (IoU > 0.5), a cluster needs ≥ 2 of 5 folds, pixels are merged by majority.

In [ ]:
per = []
for k, raw in enumerate(raws):
    o = OUT / f"test_raw_f{k}_a{ALPHA}.json"
    run(["python3", "src/vote_rescore.py", "--preds", raw, "--alpha", ALPHA, "--out", o]); per.append(o)
run(["python3", "-u", "src/ensemble.py", "--preds", *per, "--out", OUT / "test_fused.json", "--min-votes", 2])

## 4. Post‑processing and CSV
Instances are painted in descending score order (pixel‑exclusive); kept if score ≥ 0.32, painted area ≥ 200 px and at least 80 % of its own pixels are still unclaimed (retain fraction; drops fragments of already‑kept filaments). `json_to_csv.py` also validates the CSV.

In [ ]:
(OUT / "postproc.json").write_text(json.dumps({"score_thr": SCORE_THR, "min_area": MIN_AREA, "mask_thr": 0.5, "min_keep_frac": KEEP_FRAC}))
run(["python3", "src/json_to_csv.py", "--preds", OUT / "test_fused.json", "--postproc", OUT / "postproc.json", "--out", OUT / "submission.csv"])
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
print("reproduced:", md5(OUT / "submission.csv"))
print("submitted :", md5(REPO / "release" / "submission_final.csv"))
if N_IMAGES: print("(partial run: the hashes differ by design; set N_IMAGES = None for the full comparison)")

## 5. Out‑of‑fold score of this configuration
`release/oof_swa2_a0.75.json` holds the OOF predictions (the same pipeline applied to each fold's held‑out photos) that were used to choose α, the threshold and the area. The cell computes the pooled PQ with the organisers' definition (`src/pq_metric.py`); it needs the competition's training data (`FILAMENT_ROOT`) and writes `data/magfilo_singleclass.json` first if missing.

In [ ]:
if not (REPO / "data" / "magfilo_singleclass.json").exists():
    run(["python3", "src/prepare_data.py"])
run(["python3", "-u", "src/tune_postproc.py", "--preds", REPO / "release" / "oof_swa2_a0.75.json", "--fold", -1,
     "--score-thr", SCORE_THR, "--min-area", MIN_AREA, "--min-keep-frac", KEEP_FRAC, "--out", OUT / "oof_eval.json"])
print({k: round(v, 4) for k, v in json.loads((OUT / "oof_eval.json").read_text()).items() if k in ("val_pq", "val_sq", "val_rq")})
# expected: val_pq 0.4777, val_sq 0.689, val_rq 0.693